# 03: SQL analysis

**Goal:** Answer the two research questions in SQL: how far bus and MAX ridership has recovered relative to 2019, and whether ridership has recovered faster or slower than service.

**Input:** `data/processed/trimet_monthly.csv` (from notebook 02)

**Output:** Tables in `data/processed/`. Queries in `sql/`.

**Steps:** Load the cleaned data into DuckDB, run each query from a `.sql` file, check the results, and export the tables.

**Run from the repo root.**

## Views

Each query builds on the previous ones. `monthly` is the table loaded from the CSV, and everything else is a view.

| View | File | One row per | Rows | Built from |
|---|---|---|---|---|
| `core` | `00_core_view.sql` | mode, month | 186 | `monthly` |
| `baseline_month` | `01a_baseline_month.sql` | mode, month of year (2019) | 24 | `core` |
| `baseline_year` | `01b_baseline_year.sql` | mode (2019 total) | 2 | `core` |
| `monthly_recovery` | `02_monthly_recovery.sql` | mode, month | 186 | `core`, `baseline_month` |
| `rolling_12` | `03_rolling_12.sql` | mode, month (from 2019-12) | 164 | `core`, `baseline_year` |
| `upt_per_vrh` | `04_upt_per_vrh.sql` | mode, month | 186 | `core`, `rolling_12` |
| `final_table` | `05_final_table.sql` | mode, month | 186 | `core`, `monthly_recovery`, `rolling_12`, `upt_per_vrh` |

Recovery values are raw ratios (1.0 = same as 2019), not percentages.

## Setup and load 

The data is loaded into a table called `monthly`.


In [1]:
import duckdb
import pandas as pd
from pathlib import Path

CSV_PATH = "data/processed/trimet_monthly.csv"
OUT_PATH = "data/processed/recovery_monthly.csv"


def read_sql(file_path):
    return Path("sql", file_path).read_text()


con = duckdb.connect()


def run_sql(file_path):
    con.execute(read_sql(file_path))


def query(file_path):
    return con.sql(read_sql(file_path)).df()


_ = con.execute(f"""
    CREATE OR REPLACE TABLE monthly AS
    SELECT * FROM read_csv_auto('{CSV_PATH}')
    """)

## Check the load shape and data types

- Expect 506; `date`, `varchar`, `varchar`, `varchar`, `double`, `double` and the table seen in notebook 02.

In [2]:
display(con.sql("SELECT COUNT(*) AS n_rows FROM monthly").df())

display(con.sql("DESCRIBE monthly").df())

display(query("checks/load_check.sql"))

,n_rows
0,506


,column_name,column_type,null,key,default,extra
0,month,DATE,YES,None,None,None
1,mode,VARCHAR,YES,None,None,None
2,tos,VARCHAR,YES,None,None,None
3,mode_name,VARCHAR,YES,None,None,None
4,upt,DOUBLE,YES,None,None,None
5,vrh,DOUBLE,YES,None,None,None


,mode,tos,n_months,first_month,last_month
0,DR,PT,93,2019-01-01,2026-09-01
1,DR,TN,41,2023-05-01,2026-09-01
2,DR,TX,93,2019-01-01,2026-09-01
3,LR,DO,93,2019-01-01,2026-09-01
4,MB,DO,93,2019-01-01,2026-09-01
5,YR,PT,93,2019-01-01,2026-09-01


## Base view (bus and MAX only)

This analysis covers only bus (MB) and MAX light rail (LR). Each later query will start from this view called `core` that has only those two modes. 

In [3]:
run_sql("00_core_view.sql")

## Check base view

- Expect 93 per mode for both `n_rows` and `n_distinct_months`.

In [4]:
display(query("checks/00_check.sql"))

,mode,n_rows,n_distinct_months,first_month,last_month
0,LR,93,93,2019-01-01,2026-09-01
1,MB,93,93,2019-01-01,2026-09-01


## 2019 baselines

Two views will hold the baselines from 2019.
- `baseline_month`: values from each month in 2019
- `baseline_year`: total values from the 2019 calendar year

In [5]:
run_sql("01a_baseline_month.sql")
run_sql("01b_baseline_year.sql")

## Check baselines

- Expect `True`s (baseline month sum = baseline year).

In [6]:
display(con.sql("SELECT * FROM baseline_month ORDER BY mode, month_num").df())
display(con.sql("SELECT * FROM baseline_year ORDER BY mode").df())

display(query("checks/01_check.sql"))

,mode,month_num,baseline_upt,baseline_vrh
0,LR,1,3201280.0,53964.0
1,LR,2,2861440.0,49134.0
2,LR,3,3197550.0,54111.0
3,LR,4,3209820.0,53024.0
4,LR,5,3429260.0,54476.0
5,LR,6,3331050.0,52623.0
6,LR,7,3329110.0,54867.0
7,LR,8,3151630.0,51923.0
8,LR,9,3097660.0,51678.0
9,LR,10,3442530.0,54496.0


,mode,baseline_upt_year,baseline_vrh_year
0,LR,38388070.0,636933.0
1,MB,57374350.0,2084004.0


,mode,n_months,upt_match,vrh_match
0,LR,12,True,True
1,MB,12,True,True


## Monthly recovery

`monthly_recovery` will hold each month's UPT and VRH values as a share of the same month-of-the-year in the 2019 baseline.

In [7]:
run_sql("02_monthly_recovery.sql")

## Check monthly recovery

- Each core row should be kept exactly once after the join.
- Recovery values for 2019 should be 1.0 for both UPT and VRH.
- Nothing should be null, since the baseline month values are complete for 2019.
- Also hand check a few values: edge cases and a pandemic month. 

In [8]:
display(query("checks/02a_check.sql"))
display(query("checks/02b_check_spot.sql"))

,n_rows,n_core_rows,bad_2019_rows,null_rows
0,186,186,0.0,0.0


,mode,month,upt,vrh,upt_recovery_monthly,vrh_recovery_monthly
0,LR,2019-09-01,3097660.0,51678.0,1.000,1.000
1,LR,2020-04-01,1050980.0,49257.0,0.327,0.929
2,LR,2026-09-01,1880533.0,47357.0,0.607,0.916
3,MB,2019-09-01,4723280.0,170030.0,1.000,1.000
4,MB,2020-04-01,1526440.0,151215.0,0.308,0.867
5,MB,2026-09-01,3753635.0,156209.0,0.795,0.919


**Spot check results:** The checked values match the hand-computed values from the raw source file.

## Rolling 12-month recovery

`rolling_12` sums UPT and VRH over the 12 month period ending in each month. It then divides these values by the 2019 calendar year sums. The rolling sum cancels out seasonality. The recovery ratio begins with the window ending with December 2019 as it requires a full 12 month window.

In [9]:
run_sql("03_rolling_12.sql")

## Check rolling 12-month recovery

- Expect 82 rows per mode, first month 2019-12, last month 2026-09.
- December 2019 window recovery should be 1.0 as it is the baseline.
- Ensure each window covers 12 calendar months so the recovery value is valid.

In [10]:
display(query("checks/03a_check.sql"))
display(query("checks/03b_check_window.sql"))

,mode,n_rows,first_month,last_month,dec_2019_upt_recovery,dec_2019_vrh_recovery,null_rows
0,LR,82,2019-12-01,2026-09-01,1.0,1.0,0.0
1,MB,82,2019-12-01,2026-09-01,1.0,1.0,0.0


,windows_checked,bad_windows
0,164,0.0


## Trips per revenue hour

`upt_per_vrh` computes the trips (UPT) per revenue hour (VRH) on a monthly basis and over the 12 month rolling window.

In [11]:
run_sql("04_upt_per_vrh.sql")

## Check trips per revenue hour

Compare with hand calculations (UPT / VRH from the earlier spot check).

In [12]:
display(query("checks/04_check.sql"))

,mode,month,upt_per_vrh_monthly,upt_per_vrh_rolling
0,LR,2019-09-01,59.94,NaN
1,LR,2019-12-01,55.68,60.27
2,LR,2020-04-01,21.34,55.42
3,LR,2026-09-01,39.71,38.25
4,MB,2019-09-01,27.78,NaN
5,MB,2019-12-01,25.75,27.53
6,MB,2020-04-01,10.09,25.52
7,MB,2026-09-01,24.03,22.13


**Check results:** Monthly values match hand computations. Example: LR Sep 2019: 3,097,660 / 51,678 = 59.94. Rolling values are cross-checked against pandas in a later section.

## Final table

`final_table` compiles previous views. It includes the computed upt recovery, vrh recovery, rolling 12-month upt, rolling 12-month vrh, rolling 12-month upt recovery, rolling 12-month vrh recovery, upt per vrh monthly, and upt per vrh rolling. All per mode per month.

In [13]:
run_sql("05_final_table.sql")

## Check final table

- Expect `n_rows` 186, `n_unique_keys` 186, `null_monthly` 0, `null_rolling` 22, `unexpected_null_rolling` 0.

In [14]:
display(query("checks/05_check.sql"))

,n_rows,n_unique_keys,null_monthly,null_rolling,unexpected_null_rolling
0,186,186,0.0,22.0,0.0


## Cross-check rolling values in pandas

Recompute the rolling 12-month sums and recovery in pandas, from the same CSV, and compare them to `final_table`. 

- Expect 186 rows in total, 164 of them with a full window.

In [15]:
raw = pd.read_csv(CSV_PATH, parse_dates=["month"])
raw = raw[raw["mode"].isin(["MB", "LR"])].sort_values(["mode", "month"])

by_mode = raw.groupby("mode")
raw["upt_rolling_pd"] = by_mode["upt"].transform(lambda s: s.rolling(12).sum())
raw["vrh_rolling_pd"] = by_mode["vrh"].transform(lambda s: s.rolling(12).sum())

base_2019 = raw[raw["month"].dt.year == 2019].groupby("mode")[["upt", "vrh"]].sum()
raw["upt_recovery_rolling_pd"] = raw["upt_rolling_pd"] / raw["mode"].map(
    base_2019["upt"]
)
raw["vrh_recovery_rolling_pd"] = raw["vrh_rolling_pd"] / raw["mode"].map(
    base_2019["vrh"]
)

sql_df = con.sql("SELECT * FROM final_table").df()
sql_df["month"] = pd.to_datetime(sql_df["month"])
merged = sql_df.merge(raw, on=["mode", "month"], validate="one_to_one")

print(len(merged), merged["upt_rolling_pd"].notna().sum())

columns = [
    "upt_rolling",
    "vrh_rolling",
    "upt_recovery_rolling",
    "vrh_recovery_rolling",
]

for col in columns:
    sql_values = merged[col]
    pandas_values = merged[col + "_pd"]

    same_missing = (sql_values.isna() == pandas_values.isna()).all()

    max_diff = (sql_values - pandas_values).abs().max()

    print(col, same_missing, max_diff)
    assert same_missing and max_diff < 1e-6, col

print("SQL rolling values match pandas.")

186 164
upt_rolling True 0.0
vrh_rolling True 0.0
upt_recovery_rolling True 0.0
vrh_recovery_rolling True 0.0
SQL rolling values match pandas.


**Check results:** Rolling sums and recovery from SQL match the independent pandas calculation for all windows.

## Check latest month summary

- Expect recovery below 1, productivity below 2019, and the month 2026-09.

In [16]:
display(query("checks/06_check_headline.sql"))

,mode,month,upt_recovery_rolling,vrh_recovery_rolling,upt_per_vrh_rolling,upt_per_vrh_2019
0,LR,2026-09-01,0.573,0.903,38.25,60.27
1,MB,2026-09-01,0.762,0.948,22.13,27.53


## Export

Save `final_table` to `data/processed/recovery_monthly.csv` and confirm save.

- Expect shape `(186, 13)` and 22 nulls per rolling column (5).

In [17]:
out = con.sql("SELECT * FROM final_table ORDER BY mode, month").df()
out.to_csv(OUT_PATH, index=False)

saved = pd.read_csv(OUT_PATH, parse_dates=["month"])
print(saved.shape)

nulls = saved.isna().sum()
display(nulls[nulls > 0])

(186, 13)


upt_rolling             22
vrh_rolling             22
upt_recovery_rolling    22
vrh_recovery_rolling    22
upt_per_vrh_rolling     22
dtype: int64